## Modeling — Binary At-Risk (SMOTE + Bayesian Optimization, F1 At-Risk)

Gộp **Fail** và **Withdrawn** thành một lớp duy nhất: **At-risk**. Lớp còn lại là **Pass**.

Mỗi `prediction_point` train một model riêng:
1. **SMOTE** — oversample At-risk, tỷ lệ tăng `(1 + x) * N_current` với `x ∈ [0.1, 0.9]`
2. **Bayesian Optimization** — tìm đồng thời: hyperparameter mô hình + SMOTE ratio + class weight cho At-risk
3. **Metric**: F1-score của At-risk (đổi từ recall thuần vì recall-only khiến model dự đoán toàn bộ về At-risk)
4. **Evaluate** — test với `predict()` thông thường, không threshold tuning

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import RANDOM_SEED, SNAPSHOTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    recall_score, precision_score, accuracy_score, f1_score,
)

In [ ]:
data_dir = Path('../data/processed')

X_train    = pd.read_parquet(data_dir / 'X_train.parquet')
X_test     = pd.read_parquet(data_dir / 'X_test.parquet')
y_train    = pd.read_parquet(data_dir / 'y_train.parquet').squeeze()
y_test     = pd.read_parquet(data_dir / 'y_test.parquet').squeeze()
train_meta = pd.read_parquet(data_dir / 'train_meta.parquet')
test_meta  = pd.read_parquet(data_dir / 'test_meta.parquet')

# Gộp Fail (0) và Withdrawn (2) thành At-risk (1); Pass (1) -> 0
LABEL_MAP = {0: 1, 1: 0, 2: 1}
y_train = y_train.map(LABEL_MAP)
y_test  = y_test.map(LABEL_MAP)

TARGET_NAMES = ['Pass', 'At-risk']

print('X_train:', X_train.shape)
print('Features:', list(X_train.columns))
print('Snapshots:', sorted(train_meta['prediction_point'].unique()))
print('Phân phối nhãn (train):', y_train.value_counts().to_dict())

### Helper functions

In [ ]:
def get_snapshot_data(T):
    mask_tr = train_meta['prediction_point'].values == T
    mask_te = test_meta['prediction_point'].values == T
    return (
        X_train[mask_tr].reset_index(drop=True),
        X_test[mask_te].reset_index(drop=True),
        y_train[mask_tr].reset_index(drop=True),
        y_test[mask_te].reset_index(drop=True),
    )


def safe_smote(X, y, ratio_at_risk=0.5):
    """new_count = int((1 + ratio) * N_current),  ratio in [0.1, 0.9]"""
    y_s = pd.Series(y)
    n_at_risk = (y_s == 1).sum()
    n_pass    = (y_s == 0).sum()
    target = {1: int((1 + ratio_at_risk) * n_at_risk)}
    k = max(1, min(5, n_at_risk - 1, n_pass - 1))
    smote = SMOTE(sampling_strategy=target, random_state=RANDOM_SEED, k_neighbors=k)
    cols = X.columns.tolist() if hasattr(X, 'columns') else None
    X_res, y_res = smote.fit_resample(np.array(X), np.array(y))
    return pd.DataFrame(X_res, columns=cols), pd.Series(y_res)


def evaluate_full(name, y_true, y_pred):
    sep = '=' * 55
    print(f'\n{sep}\n  {name}\n{sep}')
    print(classification_report(y_true, y_pred, target_names=TARGET_NAMES, digits=3))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=TARGET_NAMES).plot(ax=ax, colorbar=False)
    ax.set_title(name)
    plt.tight_layout()
    plt.show()

---

## Vòng lặp — tất cả prediction_point

**Ước tính runtime:** ~1-2 giờ (8 mốc x 30 trials x 3 folds + 1 lần retrain mỗi mốc)

**Optuna tìm đồng thời:**
- Hyperparameter mô hình (max_iter, learning_rate, max_depth, ...)
- SMOTE ratio cho At-risk (`smote_ratio_at_risk` trong [0.1, 0.9])
- Class weight cho At-risk (`w_at_risk` trong [1, 8])

**Lưu ý:** ban đầu tối ưu thuần recall At-risk khiến model dự đoán toàn bộ là At-risk (recall=1 nhưng precision rất thấp). Đã đổi metric tối ưu sang **F1-score At-risk** để cân bằng precision/recall.

In [ ]:
models  = {}
results = []

for T in SNAPSHOTS:
    print(f'\n{"#"*60}\n  PREDICTION POINT = {T} days\n{"#"*60}')

    X_tr_raw, X_te_raw, y_tr, y_te = get_snapshot_data(T)
    X_tr = X_tr_raw.fillna(0)
    X_te = X_te_raw.fillna(0)
    print(f'Train: {len(X_tr):,} | Test: {len(X_te):,}')

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    def objective(trial):
        params = {
            'max_iter':          trial.suggest_int('max_iter', 100, 500),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth':         trial.suggest_int('max_depth', 3, 10),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 10, 100),
            'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True),
            'max_leaf_nodes':    trial.suggest_int('max_leaf_nodes', 15, 63),
            'random_state':      RANDOM_SEED,
        }
        r_at_risk = trial.suggest_float('smote_ratio_at_risk', 0.1, 0.9)
        w_at_risk = trial.suggest_float('w_at_risk', 1.0, 8.0)

        fold_scores = []
        for f_tr_idx, f_val_idx in cv.split(X_tr, y_tr):
            Xf_tr, yf_tr   = X_tr.iloc[f_tr_idx], y_tr.iloc[f_tr_idx]
            Xf_val, yf_val = X_tr.iloc[f_val_idx], y_tr.iloc[f_val_idx]

            Xf_sm, yf_sm = safe_smote(Xf_tr, yf_tr, r_at_risk)
            m = HistGradientBoostingClassifier(
                **params,
                class_weight={0: 1.0, 1: w_at_risk},
            )
            m.fit(Xf_sm, yf_sm)

            score = f1_score(yf_val, m.predict(Xf_val), pos_label=1, zero_division=0)
            fold_scores.append(score)
        return np.mean(fold_scores)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    best = study.best_params
    print(f'Best F1 At-risk CV: {study.best_value:.4f}')
    print(f'SMOTE  -> At-risk: +{best["smote_ratio_at_risk"]:.2f}x')
    print(f'Weight -> At-risk: {best["w_at_risk"]:.2f}')

    # Train final model trên full train + SMOTE với best params
    X_sm, y_sm = safe_smote(X_tr, y_tr, ratio_at_risk=best['smote_ratio_at_risk'])
    model_final = HistGradientBoostingClassifier(
        max_iter=best['max_iter'],
        learning_rate=best['learning_rate'],
        max_depth=best['max_depth'],
        min_samples_leaf=best['min_samples_leaf'],
        l2_regularization=best['l2_regularization'],
        max_leaf_nodes=best['max_leaf_nodes'],
        class_weight={0: 1.0, 1: best['w_at_risk']},
        random_state=RANDOM_SEED,
    )
    model_final.fit(X_sm, y_sm)
    models[T] = model_final

    # Evaluate
    y_pred = model_final.predict(np.array(X_te))
    evaluate_full(f'T={T}', y_te, y_pred)

    # Store metrics
    recalls    = recall_score(y_te, y_pred, average=None, labels=[0, 1], zero_division=0)
    precisions = precision_score(y_te, y_pred, average=None, labels=[0, 1], zero_division=0)
    results.append({
        'prediction_point':   T,
        'accuracy':           round(accuracy_score(y_te, y_pred), 4),
        'recall_pass':        round(recalls[0], 4),
        'recall_at_risk':     round(recalls[1], 4),
        'precision_pass':     round(precisions[0], 4),
        'precision_at_risk':  round(precisions[1], 4),
        'f1_at_risk':         round(f1_score(y_te, y_pred, pos_label=1, zero_division=0), 4),
    })

results_df = pd.DataFrame(results)
print('\n--- Tổng hợp ---')
results_df

---

## Trực quan hóa theo thời gian

In [ ]:
pts = results_df['prediction_point']

fig, ax = plt.subplots(figsize=(8, 5))
for col, label, color, ls in [
    ('recall_at_risk',    'Recall At-risk',    'tab:red',    '-'),
    ('precision_at_risk', 'Precision At-risk', 'tab:red',    '--'),
    ('f1_at_risk',        'F1 At-risk',        'purple',     '-'),
    ('recall_pass',       'Recall Pass',       'tab:green',  '-'),
    ('accuracy',          'Accuracy',          'steelblue',  '-'),
]:
    ax.plot(pts, results_df[col], marker='o', label=label, color=color, linestyle=ls, linewidth=2)
ax.set_title('Recall / Precision / F1 At-risk theo Prediction Point', fontsize=13)
ax.set_xlabel('Prediction Point (days)')
ax.set_ylabel('Score')
ax.set_xticks(SNAPSHOTS)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('HistGB + SMOTE + Bayes Opt (F1 At-Risk, binary)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Feature Importance — Permutation Importance

Tính permutation importance trên test set cho từng `prediction_point`, dùng F1-score At-risk làm scoring.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

f1_at_risk_scorer = make_scorer(f1_score, pos_label=1, zero_division=0)

importance_records = []

for T in SNAPSHOTS:
    _, X_te_raw, _, y_te = get_snapshot_data(T)
    X_te = X_te_raw.fillna(0)

    result = permutation_importance(
        models[T], X_te, y_te,
        scoring=f1_at_risk_scorer,
        n_repeats=10,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    for feat, mean_imp, std_imp in zip(X_te.columns, result.importances_mean, result.importances_std):
        importance_records.append({
            'prediction_point': T,
            'feature': feat,
            'importance_mean': mean_imp,
            'importance_std': std_imp,
        })

importance_df = pd.DataFrame(importance_records)
importance_df

In [ ]:
pivot = importance_df.pivot(index='feature', columns='prediction_point', values='importance_mean')
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Prediction Point (days)')
ax.set_title('Permutation Importance (mean drop in F1 At-Risk) theo feature x mốc thời gian')
fig.colorbar(im, ax=ax, label='Importance (mean)')
plt.tight_layout()
plt.show()

# Top feature trung bình toàn bộ mốc
top_overall = pivot.mean(axis=1).sort_values(ascending=False)
print('\nTop feature theo importance trung bình (tất cả mốc):')
print(top_overall)